# Module 2: Thread Indexing in CUDA

## Exercise 1

### Question:
If we want to use each thread in a grid to calculate one output element of a vector addition, what would be the expression for mapping the thread/block indices to the data index (`i`)?

### Choices:

A. `i = threadIdx.x + threadIdx.y;`

B. `i = blockIdx.x + threadIdx.x;`

C. `i = blockIdx.x * blockDim.x + threadIdx.x;`

D. `i = blockIdx.x * threadIdx.x;`

### Solution: **C**

*   **`blockIdx.x`**: This refers to the index of the current block within the grid. If you have multiple blocks, `blockIdx.x` tells you which block you are in.
*   **`blockDim.x`**: This represents the number of threads contained within each block. It's the dimension of the block in the x-direction.
*   **`threadIdx.x`**: This is the unique index of the current thread within its own block.

To calculate the global index `i`:
1.  `blockIdx.x * blockDim.x`: This part calculates the starting global index for the current block. It determines how many elements are covered by all the preceding blocks.
2.  `+ threadIdx.x`: This part then adds the thread's local index within its block to the starting index of the block. This ensures that each thread within that block gets a unique global index.

By combining these, each thread in the entire grid is assigned a unique, contiguous index, allowing it to correctly access and process its corresponding data element in the vector.

## Exercise 2

### Question:
Assume that we want to use each thread to calculate two adjacent elements of a vector addition. What would be the expression for mapping the thread/block indices to the data index (i) of the first element to be processed by a thread?

### Choices:

A. `i = blockIdx.x * blockDim.x + threadIdx.x * 2;`

B. `i = blockIdx.x * threadIdx.x * 2;`

C. `i = (blockIdx.x * blockDim.x + threadIdx.x) * 2;`

D. `i = blockIdx.x * blockDim.x * 2 + threadIdx.x;`

### Solution: **A**

To space out each index covered by each thread from 1 to 2 requires doubling the `threadIdx.x`. For example, if `blockIdx.x` and `blockDim.x` are both 0, then the index resulting from the calculation would be 0 for thread 0, and 2 for thread 1.

## Exercise 3

### Question:
We want to use each thread to calculate two elements of a vector addition. Each thread block processes `2 * blockDim.x` consecutive elements that form two sections. All threads in each block will process a section first, each processing one element. They will then all move to the next section, each processing one element. Assume that variable `i` should be the index for the first element to be processed by a thread. What would be the expression for mapping the thread/block indices to the data index of the first element?

### Choices:

A. `i = blockIdx.x * blockDim.x + threadIdx.x + 2;`

B. `i = blockIdx.x * threadIdx.x * 2;`

C. `i = (blockIdx.x * blockDim.x + threadIdx.x) * 2;`

D. `i = blockIdx.x * blockDim.x * 2 + threadIdx.x;`

### Solution: **D**

Each block in this scenario owns 2 times the block dimensions. Therefore, it is necessary for each thread block to skip past 2 times the number of previous blocks to prevent an overlap between indexes.

**Example:** `blockIdx = 1`, `blockDim = 3`

For `blockIdx = 1`:
*   `thread 0`: index = 6
*   `thread 1`: index = 7
*   `thread 2`: index = 8

And then for `blockIdx = 2`:
*   `thread 0`: index = 12
*   `thread 1`: index = 13
*   `thread 2`: index = 14

## Exercise 4

### Question:
For a vector addition, assume that the vector length is 8000, each thread calculates one output element, and the thread block size is 1024 threads. The programmer configures the kernel call to have a minimum number of thread blocks to cover all output elements. How many threads will be in the grid?

### Choices:

A. 8000

B. 8196

C. 8192

D. 8200

### Solution: **C**
The grid must contain a minimum of 8 blocks to contain more than 8000 threads to cover all of the output elements. Therefore, the number of threads is 1024 * 8 = 8192


## Exercise 5

### Question:
If we want to allocate an array of v integer elements in the CUDA device global memory, what would be an appropriate expression for the second argument of the `cudaMalloc` call?

### Choices:

A. n

B. v

C. n * sizeof(int)

D. v * sizeof(int)

### Solution: **D**

The second argument of the `cudaMalloc` call takes the size of the memory to allocate, therefore it is appropriate to provide the number of integers in the array multiplied by the size of an integer.


## Exercise 6

### Question:
If we want to allocate an array of n floating-point elements and have a floating-point pointer variable `A_d` to point to the allocated memory, what would be an appropriate expression for the first argument of the `cudaMalloc` call?

### Choices:

A. n

B. `(void*) A_d`

C. `*A_d`

D. `(void**) &A_d`

### Solution: **D**

`cudaMalloc`'s first argument is a pointer to a pointer, because `A_d` is a ptr we take the address of the ptr with `&A_d`. The function takes a void type ptr so we cast the ptr to the appropriate void type.

## Exercise 7

### Question:
If we want to copy 3000 bytes of data from host array `A_h` (`A_h` is a pointer to element 0 of the source array) to device array `A_d` (`A_d` is a pointer to element 0 of the destination array), what would be an appropriate API call for this data copy in CUDA?

### Choices:

A. `cudaMemcpy(3000, A_h, A_d, cudaMemcpyHostToDevice);`

B. `cudaMemcpy(A_h, A_d, 3000, cudaMemcpyDeviceToHost);`

C. `cudaMemcpy(A_d, A_h, 3000, cudaMemcpyHostToDevice);`

D. `cudaMemcpy(3000, A_d, A_h, cudaMemcpyHostToDevice);`

### Solution: **C**

The `cudaMemcpy` function takes the ptr to the array on device, then the ptr on the host, the amount of bytes being copied, and then the direction of the transfer. This corresponds with answer C.

## Exercise 8

### Question:
How would one declare a variable `err` that can appropriately receive the returned value of a CUDA API call?

### Choices:

A. `int err;`

B. `cudaError err;`

C. `cudaError_t err;`

D. `cudaSuccess_t err;`

### Solution: **C**

Cuda error types are `cudaError_t`.

## Exercise 9

Consider the following CUDA kernel and the corresponding host function that calls it:

```cuda
__global__ void foo_kernel(float* a, float* b, unsigned int N) {
    unsigned int i = blockIdx.x * blockDim.x + threadIdx.x;
    
    if (i < N) {
        b[i] = 2.7f * a[i] - 4.3f;
    }
}

void foo(float* a_d, float* b_d) {
    unsigned int N = 200000;
    foo_kernel<<<(N + 128 - 1) / 128, 128>>>(a_d, b_d, N);
}
```

a. What is the number of threads per block?

As indicated by the second argument within the `<<<>>>` braces, 128.

b. What is the number of threads in the grid?

The number of blocks is equal to `(N+128-1)/128 = (200000 + 128 -1)/128 = 1563`.
Because there are 128 threads in each block, there is a total of `1563 * 128 = 200064` threads.

c. What is the number of blocks in the grid?

Per above, 1563.

d. What is the number of threads that execute the code on line 02?

All `2000064` threads will execute the line because each thread will enter the function and reach that line.

e. What is the number of threads that execute the code on line 04?

Only the first `200000` threads will execute line 04 due to the if statement limiting execution to only the number of elements specified by the host kernel.


## Exercise 10:

### Question:
A new summer intern was frustrated with CUDA. He has been complaining that CUDA is very tedious. He had to declare many functions that he plans to execute on both the host and the device twice, once as a host function and once as a device function. What is your response?

### Solution: **A**
A) The functions that need to be duplicated can be compiled with both the `__host__` and `__device__` compiler directives to minimize code duplication.